#  Symbolic Regression

:::{admonition} [Yasmeen Asali](https://orcid.org/0000-0002-8320-2198) (Yale University)
:class: Author

*Description*: Text for exercise description goes here.

*Intended Audience*: Early Undergraduate

*tags*: `concept`,`concept`,

*Requirements*: `astropy`, `numpy`

*Last Updated: August 22, 2025*
:::

:::{admonition} Learning Objectives
:class: learningobjective

1. Learning objective 1
2. Learning objective 2

:::

## Photometric Redshifts

One of the most fundamental measurements in astronomy is the redshift of a galaxy, which tells us how fast it is moving away due to the expansion of the Universe and allows us to estimate its distance. The most precise way to measure a galaxy's redshift is through spectroscopy, where absorption or emission lines can be directly observed and shifted in wavelength. However, spectroscopy requires long exposure times on large telescopes, making it impractical to obtain spectra for the millions of galaxies detected in modern imaging surveys.

An alternative approach is to use photometric redshifts (photo-z's). These estimates rely on the fact that a galaxy's broadband colors (for example, its brightness in filters $u$, $g$, $r$, $i$, $z$) are correlated with its redshift. The galaxy's spectrum gets shifted through the filters in a systematic way: nearby galaxies look bluer, while distant galaxies appear redder as features like the 4000 $\AA$ break move across the bands. Photometric redshifts are less precise than spectroscopic redshifts, but they can be measured for orders of magnitude more objects, enabling cosmological analyses that require very large samples.

Traditionally, photometric redshifts have been estimated using either:
- Template fitting, where model galaxy spectra are shifted to different redshifts and compared to observed colors, or
- Empirical methods, where statistical or machine-learning models learn a mapping between photometry and spectroscopic redshifts in a training sample.

In this exercise, we will explore a more recent approach: symbolic regression. Symbolic regression is a machine-learning technique that searches over mathematical expressions to find analytic formulas that approximate the relationship between input variables and a target. Instead of training a "black box" model, symbolic regression produces an explicit equation, which can provide both predictive power and physical insight.

Here, we will use symbolic regression to estimate galaxy redshifts from their optical magnitudes and colors. The goal is to see whether the algorithm can recover a compact mathematical expression that relates photometry to redshift and to evaluate how well this performs compared to the true spectroscopic redshifts in our dataset.

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('sdss_photz.csv')


In [6]:
df['zred'].describe()

count    4072.000000
mean        0.409952
std         0.232235
min         0.004953
25%         0.204167
50%         0.417340
75%         0.607680
max         0.978773
Name: zred, dtype: float64

This paper uses CNN+ANN to create a new/updated photo-z measurement code: https://ui.adsabs.harvard.edu/abs/2025A%26A...698A.276J/abstract 
This paper uses symbolic regression to derive an expression for photozs from optical colors from SDSS: https://ui.adsabs.harvard.edu/abs/2014MNRAS.443L..34K/abstract   

In [7]:
df['u_g'] = df['u'] - df['g']
df['g_r'] = df['g'] - df['r']
df['r_i'] = df['r'] - df['i']
df['i_z'] = df['i'] - df['z']
df['g_i'] = df['g'] - df['i']
df['u_r'] = df['u'] - df['r']

feature_cols = ['u','g','r','i','z','u_g','g_r','r_i','i_z','g_i','u_r']
X = df[feature_cols].values
y = df['zred'].values

# ----- 3) Stratified train/test split -----
# Bin redshifts for stratification
bins = np.linspace(y.min(), y.max(), 12)
y_bins = pd.cut(y, bins=bins, labels=False, include_lowest=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y_bins
)

https://github.com/MilesCranmer/PySR 

::::{admonition} Exercise
:class: tip

1. Prepare data  
   - Read `sdss_photz.csv` into a DataFrame.  
   - Add columns corresponding to optical colors (e.g., `u-g`, `g-r`, `r-i`, `i-z`, `g-i`, `u-r`).  
   :::{hint}
   Colors capture *spectral shape* and track features like the 4000$\AA$ break across filters, which can help improve photo-z performance compared to magnitudes alone.
   :::
   - Split data into train/test (e.g., 80/20) 

2. Run symbolic regression  
   - Use a symbolic regression estimator with an sklearn-like API (e.g., `gplearn.SymbolicRegressor`).  
   - Choose a function set (e.g., add, sub, mul, div, sqrt, log, abs, inv).  
   - Train on the **training set only**.

3. **Evaluation & Plots**  
   - Compute and report: MAE, RMSE, $R^2$, bias (mean $\Delta z$), and $\sigma_{\mathrm{NMAD}} = 1.48\,\mathrm{median}\!\left(\frac{| \Delta z - \mathrm{median}(\Delta z) |}{1+z_{\mathrm{spec}}}\right)$.  
   - Define $\Delta z = z_{\mathrm{pred}} - z_{\mathrm{spec}}$, and the outlier fraction as $|\Delta z|/(1+z_{\mathrm{spec}}) > 0.15$ (or a threshold of your choosing).  
   - Make 2 plots:
     1. **Predicted vs. True** redshift with the 1:1 line.
     2. **Residuals vs. True** redshift.
   - Print the learned analytic expression.  
   - Briefly comment on which colors/magnitudes appear and whether the form seems physically sensible (e.g., dependence on `g-r` or `r-i`).

::::
